In [4]:
import numpy as np
import casadi as ca
import torch
from Utilities.block_builders import *
from Networks.networks import PINN, AlgNN

pinn = PINN(
    hidden_units=[64,64,64]
)

algnn = AlgNN(
    hidden_units=[64,64,64,64]
)


# 1) Load your torch models
# from your_code import PINN, AlgNN, load_model_weights
pinn = load_model_weights(pinn,"Training/PINN.pth")
algnn  = load_model_weights(algnn,"Training/AlgNN.pth")

# 2) Extract weights
pinn_w = extract_pinn_standard_weights(pinn)
alg_w  = extract_algnn_standard_weights(algnn)

# 3) IMPORTANT: use the SAME scaling constants as the trained models
# You can read them from buffers:
pinn_y_min = pinn.y_min.cpu().numpy().tolist()
pinn_y_max = pinn.y_max.cpu().numpy().tolist()
pinn_u_min = pinn.u_min.cpu().numpy().tolist()
pinn_u_max = pinn.u_max.cpu().numpy().tolist()

alg_y_min = algnn.y_min.cpu().numpy().tolist()
alg_y_max = algnn.y_max.cpu().numpy().tolist()
alg_u_min = algnn.u_min.cpu().numpy().tolist()
alg_u_max = algnn.u_max.cpu().numpy().tolist()
alg_z_min = algnn.z_min.cpu().numpy().tolist()
alg_z_max = algnn.z_max.cpu().numpy().tolist()

F_u2z = build_casadi_surrogate_u2z(
    pinn_weights=pinn_w,
    algnn_weights=alg_w,
    pinn_y_min=pinn_y_min, pinn_y_max=pinn_y_max,
    pinn_u_min=pinn_u_min, pinn_u_max=pinn_u_max,
    alg_y_min=alg_y_min,   alg_y_max=alg_y_max,
    alg_u_min=alg_u_min,   alg_u_max=alg_u_max,
    alg_z_min=alg_z_min,   alg_z_max=alg_z_max,
)

# 4) Differentiate
u_sym = ca.MX.sym("u", 2)
z_expr = F_u2z(u=u_sym)["z"]
J = ca.jacobian(z_expr, u_sym)
J_fun = ca.Function("J_u2z", [u_sym], [J])

# 5) Test numeric
u_test = np.array([1.0, 1.0])
print("z(u) =", F_u2z(u=u_test)["z"])
print("dz/du =", J_fun(u_test))

=== LOADING MODEL FROM Training/PINN.pth ===
=== LOADING MODEL FROM Training/AlgNN.pth ===
z(u) = [17.1447, 85.0679, 79.0553]
dz/du = 
[[1.00591, 1.11715], 
 [-4.57533, -3.7785], 
 [-4.55161, -4.32023]]
